# 3D Vision — Point Clouds & NeRFs Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: PointNet classifier

In [ ]:
```python

import torch

import torch.nn as nn

class PointNet(nn.Module):

    def __init__(self, num_classes=10):

        super().__init__()

        self.mlp1 = nn.Sequential(

            nn.Conv1d(3, 64, 1),    nn.BatchNorm1d(64),   nn.ReLU(inplace=True),

            nn.Conv1d(64, 64, 1),   nn.BatchNorm1d(64),   nn.ReLU(inplace=True),

        )

        self.mlp2 = nn.Sequential(

            nn.Conv1d(64, 128, 1),  nn.BatchNorm1d(128),  nn.ReLU(inplace=True),

            nn.Conv1d(128, 1024, 1), nn.BatchNorm1d(1024), nn.ReLU(inplace=True),

        )

        self.head = nn.Sequential(

            nn.Linear(1024, 512),   nn.BatchNorm1d(512),  nn.ReLU(inplace=True),

            nn.Dropout(0.3),

            nn.Linear(512, 256),    nn.BatchNorm1d(256),  nn.ReLU(inplace=True),

            nn.Dropout(0.3),

            nn.Linear(256, num_classes),

        )

    def forward(self, x):

        # x: (N, 3, num_points) — transposed for Conv1d

        x = self.mlp1(x)

        x = self.mlp2(x)

        x = torch.max(x, dim=-1)[0]       # (N, 1024)

        return self.head(x)

pts = torch.randn(4, 3, 1024)

net = PointNet(num_classes=10)

print(f"output: {net(pts).shape}")

print(f"params: {sum(p.numel() for p in net.parameters()):,}")

In [ ]:
```

About 1.6M parameters. Runs on 1,024 points per cloud.

### Step 2: Positional encoding

In [ ]:
```python

def positional_encoding(x, L=10):

    """

    x: (..., D) -> (..., D * 2 * L)

    """

    freqs = 2.0 ** torch.arange(L, dtype=x.dtype, device=x.device)

    args = x.unsqueeze(-1) * freqs * 3.141592653589793

    sinc = torch.cat([args.sin(), args.cos()], dim=-1)

    return sinc.reshape(*x.shape[:-1], -1)

x = torch.randn(5, 3)

y = positional_encoding(x, L=10)

print(f"input:  {x.shape}")

print(f"encoded: {y.shape}     # (5, 60)")

In [ ]:
```

Multiplying by `2^l * pi` gives progressively higher frequencies.

### Step 3: Tiny NeRF MLP

In [ ]:
```python

class TinyNeRF(nn.Module):

    def __init__(self, L_pos=10, L_dir=4, hidden=128):

        super().__init__()

        self.L_pos = L_pos

        self.L_dir = L_dir

        pos_dim = 3 * 2 * L_pos

        dir_dim = 3 * 2 * L_dir

        self.trunk = nn.Sequential(

            nn.Linear(pos_dim, hidden), nn.ReLU(inplace=True),

            nn.Linear(hidden, hidden),  nn.ReLU(inplace=True),

            nn.Linear(hidden, hidden),  nn.ReLU(inplace=True),

            nn.Linear(hidden, hidden),  nn.ReLU(inplace=True),

        )

        self.sigma = nn.Linear(hidden, 1)

        self.color = nn.Sequential(

            nn.Linear(hidden + dir_dim, hidden // 2), nn.ReLU(inplace=True),

            nn.Linear(hidden // 2, 3), nn.Sigmoid(),

        )

    def forward(self, x, d):

        x_enc = positional_encoding(x, self.L_pos)

        d_enc = positional_encoding(d, self.L_dir)

        h = self.trunk(x_enc)

        sigma = torch.relu(self.sigma(h)).squeeze(-1)

        rgb = self.color(torch.cat([h, d_enc], dim=-1))

        return sigma, rgb

nerf = TinyNeRF()

x = torch.randn(128, 3)

d = torch.randn(128, 3)

s, c = nerf(x, d)

print(f"sigma: {s.shape}   rgb: {c.shape}")

In [ ]:
```

Tiny compared to the original NeRF (which has 2 MLP trunks of depth 8). Enough to demonstrate the architecture.

### Step 4: Volumetric rendering along a ray

In [ ]:
```python

def volumetric_render(sigma, rgb, t_vals):

    """

    sigma: (..., N_samples)

    rgb:   (..., N_samples, 3)

    t_vals: (N_samples,) distances along the ray

    """

    delta = torch.cat([t_vals[1:] - t_vals[:-1], torch.full_like(t_vals[:1], 1e10)])

    alpha = 1.0 - torch.exp(-sigma * delta)

    trans = torch.cumprod(torch.cat([torch.ones_like(alpha[..., :1]), 1.0 - alpha + 1e-10], dim=-1), dim=-1)[..., :-1]

    weights = alpha * trans

    rendered = (weights.unsqueeze(-1) * rgb).sum(dim=-2)

    depth = (weights * t_vals).sum(dim=-1)

    return rendered, depth, weights

N = 64

t_vals = torch.linspace(2.0, 6.0, N)

sigma = torch.rand(N) * 0.5

rgb = torch.rand(N, 3)

rendered, depth, weights = volumetric_render(sigma, rgb, t_vals)

print(f"rendered colour: {rendered.tolist()}")

print(f"depth:           {depth.item():.2f}")

In [ ]:
```

One ray, 64 samples, composite to a single RGB pixel and a depth.

## Exercises

In [ ]:
1. **(Easy)** Show that PointNet is permutation-invariant: run the same cloud through twice, once with points shuffled. Verify outputs are identical up to floating-point noise.
2. **(Medium)** Implement a minimal ray-generation function that, given camera intrinsics and pose, produces ray origins and directions for every pixel of an H x W image.
3. **(Hard)** Train a TinyNeRF on a synthetic dataset of rendered views of a coloured cube (generated via differentiable rendering or a simple ray tracer). Report rendering loss at epoch 1, 10, and 100. At what epoch does the model produce recognisable views?